In [ ]:
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, MicroSAMMerger, MicroSAMMergerConfig
from vistiq.segment import TiledSegmentationFlow, TiledSegmentationFlowConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.io import ImageLoader, ImageLoaderConfig
from vistiq.core import FuncProcessor, FuncProcessorConfig, Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import ProcessChain, ProcessChainConfig, ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.utils import ArrayIteratorConfig 
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector
from vistiq.io import ImageWriterConfig, ImageWriter

import stackview
import os
import numpy as np
import math
import logging

# Configure logger and check availability of accelerators

In [ ]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

import torch
logger.info(f"Torch version: {torch.__version__}. Cuda available: {torch.cuda.is_available()}, MPS available: {torch.backends.mps.is_available()}")

# Load image

In [ ]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
#path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"
path="Animal 1.lif"
#path="/Users/khs3z/Documents/SDS_/projects/Siegrist/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Animal 1.lif"

scene_index = 0

embedding_path = "./embeddings"
#embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [ ]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=True,
    #substack="C:1"
)
img, metadata = ImageLoader(ilc).run(path)

In [ ]:
if "C" in metadata["axes"]:
    vimg = np.concatenate(np.unstack(img, axis=0), axis=-1)
else:
    vimg = img
stackview.slice(vimg)
#stackview.switch(img, colormap=["pure_green", "pure_blue", "pure_red"], toggleable=True)

# Preprocess

In [ ]:
ppcfg = ProcessChainConfig(
    processors = [
        RescaleConfig(
            low=2, 
            high=98, 
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.filters.gaussian",
            kwargs={"sigma": 1.0},
            iterator_config=ArrayIteratorConfig(slice_def=(-2,-1)) # YX over each Z-plane and channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_gamma",
            kwargs={"gamma": 0.2},
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="skimage.exposure.adjust_sigmoid",
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        RescaleConfig(
            dtype=np.uint8, 
            iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1)) # ZYX over each channel
        ),
        FuncProcessorConfig(
            func="numpy.max", 
            kwargs={"axis":("C")}, # Project all channels into one
            strict_axis=False,     # don't throw exception if the input is a single channel image already
            dtype=np.uint16,
        ),
    ]
)
c_img, c_metadata = ProcessChain(ppcfg).run(img, metadata=metadata, workers=-1)
metadata, c_metadata

In [ ]:
stackview.slice(c_img)

# Segment

In [ ]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mscfg)


rfcfg = RegionFilterConfig(
    filters=[
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area", 
                range=(4000, np.inf)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="aspect_ratio", 
                range=(0.5, 1.0)
            )
        ),
    ]
)

rf = RegionFilter(rfcfg)

tsfcfg = TiledSegmentationFlowConfig(
    segmenter = microsam,
    region_filter = rf,
    tile_factor=(5,5),
    resize_factor=(0.25,0.25),
    iou_threshold=0.7,
    consensus_threshold=0.3,
)
labels, t_labels, t_masks, untiled, t_proj = TiledSegmentationFlow(tsfcfg).run(c_img, metadata=c_metadata, verbose=0, config=tsfcfg)

In [ ]:
print (f"Unique labels (incl. background): {np.unique(labels)}")

In [ ]:
vlabels = np.concatenate(len(metadata["channel_names"])*[labels], axis=-1)
stackview.blend(vimg.astype("uint16"), vlabels.astype("uint64"), blend_factor=40)

# Analyze regions

In [ ]:
racfg = RegionAnalyzerConfig(
    properties=["volume", "cross_sectional_area", "bbox", "aspect_ratio"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

measurements = ra.run(labels, metadata=c_metadata)

In [ ]:
measurements

# Save label

In [ ]:
c_metadata["channel_names"] = ["Brain Lobes"]

In [ ]:
imc = ImageWriterConfig(overwrite=True)
outpath = ".".join(path.split(".")[:-1]) + f"-scene-{scene_index}.tif"
ImageWriter(imc).run(labels, outpath, metadata=c_metadata)

# Segment Cells

In [ ]:
from vistiq.io import unstack_image
from prefect import flow
from prefect.task_runners import ProcessPoolTaskRunner
from prefect.futures import wait

@flow(task_runner=ProcessPoolTaskRunner(max_workers=8))
def segment_cells(img, metadata) -> list[np.ndarray]:
    (unstacked_img, unstacked_metadata) = unstack_image(img, metadata, axis="C")

    mscfg = MicroSAMSegmenterConfig(
        iterator_config=ArrayIteratorConfig(slice_def=()),
        embedding_path=embedding_path,
    )
    microsam = MicroSAMSegmenter(mscfg)

    rfcfg = RegionFilterConfig(
        filters=[
            RangeFilter(
                RangeFilterConfig(
                    attribute="cross_sectional_area", 
                    range=(10, 500)
                )
            ),
            RangeFilter(
                RangeFilterConfig(
                    attribute="aspect_ratio", 
                    range=(0.5, 1.0)
                )
            ),
        ]
    )
    rf = RegionFilter(rfcfg)

    sfcfg = SegmentationFlowConfig(
        segmenter = microsam,
        region_filter = rf,
    )
    futures = SegmentationFlow(sfcfg).run.map(unstacked_img, metadata=unstacked_metadata)
    results = [f.result() for f in futures]
    return results

cell_labels = segment_cells(img, metadata)

In [ ]:
stackview.slice(np.concatenate(cell_labels, axis=-1))

In [ ]:
rgb_swp_cfg = FuncProcessorConfig(
    func="numpy.moveaxis",
    args=[0,3],
)
rgb_img, rgb_metadata = FuncProcessor(rgb_swp_cfg).run(img, metadata=metadata)

In [ ]:
mscfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mscfg)

rfcfg = RegionFilterConfig(
    filters=[
        RangeFilter(
            RangeFilterConfig(
                attribute="cross_sectional_area", 
                range=(10, 500)
            )
        ),
        RangeFilter(
            RangeFilterConfig(
                attribute="aspect_ratio", 
                range=(0.5, 1.0)
            )
        ),
    ]
)
rf = RegionFilter(rfcfg)

sfcfg = SegmentationFlowConfig(
    segmenter = microsam,
    region_filter = rf,
)

#rgb_cell_labels = SegmentationFlow(sfcfg).run(rgb_img[40])